# 从零实现 RLM：用递归语言模型回答 CAE 论文问题

本 notebook 自底向上手写一个最小可跑的 **Recursive Language Model (RLM)**
（参考 arXiv:2512.24601 / [fast-rlm](https://github.com/avbiswas/fast-rlm)），
并用它回答 `/home/juli/RLM/CAE-MDs` 里 8 篇关于流固耦合 / 水下爆炸 /
LS-DYNA 的 CAE 论文的问题。

每一节加一个组件、跑一次 demo，对比上一节的差距。版本进展：

| 版本 | 加什么 | 跑什么 demo |
|---|---|---|
| v1 | REPL + `FINAL` + 主循环 | 列出所有论文 paper_id |
| v2 | 加 `await llm_query(...)`（**异步、递归**） | 抽取单篇论文的关键字段 |
| v3 | 加 `search_kb` 关键词工具 | 检索 + 多篇对比 |
| §8 | 端到端真实问答 | 两个非平凡问题 |

> 阅读路径：§1 → §4 → §5 → §6 → §7（实战）。想直接看效果跳到 §7。

> ⚠️ **安全**：本 notebook 用同进程 `exec()` 跑 LLM 写的 Python 代码，
> **没有沙箱**。LLM 在 REPL 里能 `import os; os.system(...)`。仅在自己的机器上跑，
> 不要把它作为服务暴露给不可信用户。

## §0 运行前提

1. 在本 notebook 同目录放 `.env`：
   ```
   OPENAI_BASE_URL=https://dubrify.com/v1
   OPENAI_API_KEY=sk-...
   OPENAI_MODEL=deepseek-v4-flash
   ```
2. 安装 `openai>=1`、`python-dotenv`。
3. 知识库 `/home/juli/RLM/CAE-MDs` 已包含 8 个 `.md` 文件。


In [4]:
import os, re, json, asyncio, ast, io, contextlib, traceback, pathlib, time, textwrap
from dataclasses import dataclass, field
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI, AsyncOpenAI

# 从 notebook 同目录的 .env 加载凭据
load_dotenv(pathlib.Path(".env").resolve())

BASE_URL = os.environ["OPENAI_BASE_URL"]
API_KEY = os.environ["OPENAI_API_KEY"]
MODEL = os.environ["OPENAI_MODEL"]

# DeepSeek v4 是 reasoning model：默认思考阶段会让每次 RLM 迭代慢 1–3 秒并
# 烧掉看不见的 reasoning tokens。dubrify 后端识别此 flag，需要打开。
LLM_KWARGS = {
    "temperature": 0.8,
    "extra_body": {"thinking": {"type": "disabled"}},
}

sync_client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
async_client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)

def chat(messages: list[dict]) -> tuple[str, dict]:
    """同步单次 LM 调用。返回 (content, usage_dict)。"""
    resp = sync_client.chat.completions.create(model=MODEL, messages=messages, **LLM_KWARGS)
    u = resp.usage
    return resp.choices[0].message.content or "", {
        "prompt_tokens": u.prompt_tokens, "completion_tokens": u.completion_tokens,
    }

async def achat(messages: list[dict]) -> tuple[str, dict]:
    """异步单次 LM 调用。RLM 主循环要在 async 上下文里跑，因此用 async client。"""
    resp = await async_client.chat.completions.create(model=MODEL, messages=messages, **LLM_KWARGS)
    u = resp.usage
    return resp.choices[0].message.content or "", {
        "prompt_tokens": u.prompt_tokens, "completion_tokens": u.completion_tokens,
    }

# 烟雾测试
content, usage = chat([{"role": "user", "content": "Reply with: PONG"}])
print(f"LLM says: {content!r}   usage: {usage}")


LLM says: 'PONG'   usage: {'prompt_tokens': 9, 'completion_tokens': 2}


## §1 RLM 是什么

RLM 的核心观察：

- 一个普通 LLM 看不到的、"远大于上下文窗口"的提示，可以让 LLM 通过一个 REPL
  与之**交互**——它写 Python 代码去 grep、切片、调用子 LLM，最后调用一个
  `FINAL(x)` 把答案交回。
- 子 LLM 调用的结果**不会自动塞进父 LLM 的对话历史**——它只是 REPL 里一个
  Python 变量，父 LLM 想看才 `print(...)`。这就把上下文与中间状态彻底解耦。

因此 RLM ≈ **一个 LLM + 一个 Python 解释器 + 一个能递归 spawn 自己的子例程**。
我们要手写的就是这三件事：
1. 一个能 `exec` 代码的 REPL，命名空间持久化跨步保留
2. 一个 `FINAL(value)` 完成信号
3. 一个 `await llm_query(context)` 异步递归调用子 RLM 的函数


## §2 加载 CAE-MDs 知识库

知识库路径 `/home/juli/RLM/CAE-MDs`，含 8 篇 markdown：
- 1 本英文专著（Benson 的 ALE/FSI 综述，约 100k 字符）
- 1 篇 thyssenkrupp 的 FRP 水下冲击会议论文（英文）
- 6 篇中文论文/学位论文（水下爆炸、加筋结构破损、LS-DYNA 应用等）

读进 `dict[paper_id, full_text]`，`paper_id` 就是文件名 stem。


In [5]:
KB_DIR = pathlib.Path("/home/juli/RLM/CAE-MDs")
papers: dict[str, str] = {p.stem: p.read_text(encoding="utf-8") for p in sorted(KB_DIR.glob("*.md"))}

print(f"Loaded {len(papers)} papers:")
for pid, text in papers.items():
    print(f"  - {pid[:70]:70s}  {len(text):>8,} chars")
print(f"\nTotal: {sum(len(t) for t in papers.values()):,} chars")


Loaded 8 papers:
  - Arbitrary_Lagrangian-Eulerian_and_Fluid-Structure Interaction Numerica   554,647 chars
  - PhD 基于通用程序的水下爆炸及其对结构作用的数值模拟研究                                            122,807 chars
  - oezarmut_thyssenkrupp_Fluid-Composite Structure-Interaction in Underwa    33,578 chars
  - 不同加筋结构在水中接触爆炸下的破损规律                                                       10,623 chars
  - 基于ANSYS_LS-DYNA的钢板混凝土墙冲击实验的有限元分析                                          14,403 chars
  - 基于LS-DYNA的液电效应冲击波数值模拟                                                     28,648 chars
  - 基于LS-DYNA的高速破片水中运动特性流固耦合数值模拟                                               8,886 chars
  - 水下爆炸冲击荷载作用下混凝土重力坝的破坏模式                                                    13,035 chars

Total: 786,627 chars


In [6]:
def rough_tokens(s: str) -> int:
    """粗略估算：中文 1 char ≈ 1 token，英文按 4 char/token。"""
    cn = sum(1 for c in s if "\u4e00" <= c <= "\u9fff")
    return cn + (len(s) - cn) // 4

for pid, text in papers.items():
    print(f"  ~{rough_tokens(text):>7,} tok  {pid[:60]}")
print(f"\nTotal ~{sum(rough_tokens(t) for t in papers.values()):,} tokens")


  ~138,661 tok  Arbitrary_Lagrangian-Eulerian_and_Fluid-Structure Interactio
  ~ 77,871 tok  PhD 基于通用程序的水下爆炸及其对结构作用的数值模拟研究
  ~  8,394 tok  oezarmut_thyssenkrupp_Fluid-Composite Structure-Interaction 
  ~  4,725 tok  不同加筋结构在水中接触爆炸下的破损规律
  ~  5,880 tok  基于ANSYS_LS-DYNA的钢板混凝土墙冲击实验的有限元分析
  ~ 12,127 tok  基于LS-DYNA的液电效应冲击波数值模拟
  ~  4,183 tok  基于LS-DYNA的高速破片水中运动特性流固耦合数值模拟
  ~  5,765 tok  水下爆炸冲击荷载作用下混凝土重力坝的破坏模式

Total ~257,606 tokens


> 全语料约 ~30 万 token，没有任何主流 LLM 能一次塞下。下一节先做个反例感受一下，
> 然后从 §4 开始动手实现 RLM。


## §3 反例：直接把整个语料塞给 LLM

最 naive 的做法："把所有论文连接起来，问一个问题，让 LLM 一次回答"。
我们故意试一下，看会发生什么——通常是 `context_length_exceeded` 或答非所问。


In [7]:
question = "在 Benson 关于 ALE/FSI 的论文中，作者主张 fully Lagrangian 与 fully Eulerian 各自的优缺点是什么？"
big_blob = "\n\n=== PAPER ===\n\n".join(f"[{pid}]\n{text}" for pid, text in papers.items())
print(f"Naive prompt size: ~{rough_tokens(big_blob):,} tokens")

try:
    content, usage = chat([
        {"role": "system", "content": "你是 CAE 文献助手，根据下方语料用中文作答。"},
        {"role": "user", "content": question + "\n\n语料：\n" + big_blob},
    ])
    print("--- LLM RESPONSE ---")
    print(content[:1200])
    print(f"\n--- USAGE: {usage}")
except Exception as e:
    print(f"--- FAILED: {type(e).__name__}: {str(e)[:300]}")


Naive prompt size: ~257,819 tokens
--- LLM RESPONSE ---
根据提供的语料，在 Benson 关于 ALE/FSI 的论文中（第一章），作者主张 fully Lagrangian 与 fully Eulerian 各自的优缺点如下：

**Fully Lagrangian 方法：**
1.  **优点**：计算网格随材料一起移动，因此非常经济，并且能够非常精确地分辨材料边界和自由表面。
2.  **缺点**：变形必须有限，否则网格的畸变会导致计算不精确和数值不稳定。

**Fully Eulerian 方法：**
1.  **优点**：使用固定网格消除了对材料变形程度的限制。
2.  **缺点**：引入了与材料通过网格输运相关的对流项的额外复杂性，需要额外的计算时间和处理。

--- USAGE: {'prompt_tokens': 291496, 'completion_tokens': 149}


> 预期：要么 API 抛 context length 错误（"This model's maximum context length is..."），
> 要么调用返回但答得很差/数字编造。
>
> 现在动手写 RLM。


## §4 RLM v1：最小可跑（REPL + FINAL + 主循环）

三件事：

### 4.1 REPL —— 一个能跑代码、保留命名空间、支持 `await` 的盒子

我们用 Python 标准库的 `compile(..., flags=ast.PyCF_ALLOW_TOP_LEVEL_AWAIT)`
让 REPL 能跑带 `await` 的代码——这是为 §5 的 `await llm_query(...)` 做准备。

REPL 还要做的事：
- 暴露一个 `FINAL(value)` 函数：模型一调它，主循环就退出并返回 `value`
- 捕获 stdout/stderr 并截断到末尾 N 字符（避免历史爆炸）


In [8]:
@dataclass
class REPL:
    """In-process Python REPL with persistent namespace, FINAL hook, and async support."""
    ns: dict = field(default_factory=dict)
    done: bool = False
    final: Any = None
    truncate_len: int = 2000

    def __post_init__(self):
        def FINAL(value):
            self.final = value
            self.done = True
        self.ns["FINAL"] = FINAL

    async def execute(self, code: str) -> str:
        """跑一段代码（可含 top-level await），返回截断后的 stdout/stderr。"""
        buf = io.StringIO()
        try:
            with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                co = compile(code, "<repl>", "exec", flags=ast.PyCF_ALLOW_TOP_LEVEL_AWAIT)
                result = eval(co, self.ns, self.ns)
                if asyncio.iscoroutine(result):
                    await result
        except Exception:
            buf.write("\n" + traceback.format_exc())
        out = buf.getvalue()
        if len(out) > self.truncate_len:
            return f"[TRUNCATED: last {self.truncate_len} chars]\n...{out[-self.truncate_len:]}"
        return out or "[EMPTY OUTPUT — remember to print() what you want to see]"


### 4.2 系统提示词

直接使用 fast-rlm 论文实现里的 system prompt（去掉了 JS 转义）。要点：

- `context` 是 REPL 里已经存在的变量
- 唯一允许的代码块格式是 ```` ```repl ... ``` ````
- 完成时调 `FINAL(...)`，**不要**写成 `FINAL("variable_name")`
- 一轮回复只允许一个代码块

v1 系统提示先不提 `llm_query`——下一节再加。


In [9]:
SYSTEM_V1 = """\
You are tasked with answering a query with associated context. You can access, transform, and analyze this context interactively in a REPL environment. You will be queried iteratively until you provide a final answer.

You will be provided with information about your context by the user.
This metadata will include the context type, total characters, etc.

The REPL environment is initialized with:

1. A `context` variable that contains extremely important information about your query. You should check the content of the `context` variable to understand what you are working with.

   `context` may be a Python string OR a Python dict. The initial probe will tell you which.
   When `context` is a dict, prefer indexing directly (e.g. `context["episodes"]`) instead of stringifying it.

2. A global function FINAL which you can use to return your answer as any native Python value (string, dict, list, int, float, etc.).

You interact with the REPL by writing Python code wrapped in triple backticks with the `repl` language identifier:

```repl
chunk = context[:10000]
print(chunk)
```

Rules:
- print() statements show output to you (they will be truncated to the last 2000 chars).
- Just typing a variable name does NOT print it; use print().
- The REPL is persistent across turns — past variables remain.
- Do NOT use FINAL("variable_name") — that returns the literal string "variable_name". Pass the variable directly: FINAL(answer).
- Output exactly ONE ```repl``` block per turn.
- This is multi-turn. You do NOT have to call FINAL on the first turn. Inspect things first.

When you are done, call FINAL(your_answer) inside the repl block.

* WHAT IS BAD *
Trying to print() the whole context to "see it all" is a sign of low intelligence. Use Python to extract structure, slice, search.

* KNOWING WHEN TO QUIT *
Time is ticking every step. If you cannot finish, return what you know honestly.

Your expected response format is exactly:
```repl
your python code
FINAL(answer)
```
"""


### 4.3 主循环

每轮：
1. 拼当前 history 调一次 LLM → 模型回一段文字（含 `` ```repl ... ``` `` 块）
2. 抽出代码块，扔进 REPL 跑
3. 若 `repl.done`，return；否则把"模型回复 + REPL 输出"两条消息 push 回 history 进入下一轮

教学版用 step 数硬截。


In [10]:
def extract_repl(text: str) -> str | None:
    """从 LLM 输出里抽 ```repl ... ``` 块。"""
    m = re.search(r"```repl\s*\n(.*?)```", text, re.DOTALL)
    return m.group(1).strip() if m else None

async def run_rlm_v1(question: str, context: Any, max_steps: int = 8, verbose: bool = True) -> dict:
    repl = REPL(ns={"context": context})
    probe = _make_probe(context)
    history = [
        {"role": "system", "content": SYSTEM_V1},
        {"role": "user", "content": f"任务：{question}\n\nREPL 初始 probe:\n{probe}"},
    ]
    total = {"prompt_tokens": 0, "completion_tokens": 0}
    for step in range(max_steps):
        content, usage = await achat(history)
        for k in total: total[k] += usage[k]
        if verbose:
            print(f"\n=== v1 step {step} ===\n{content}\n")
        code_str = extract_repl(content)
        if code_str is None:
            history += [
                {"role": "assistant", "content": content},
                {"role": "user", "content": "你的回复里没有 ```repl``` 代码块。请重写一个。"},
            ]
            continue
        stdout = await repl.execute(code_str)
        if verbose:
            print(f"--- REPL ---\n{stdout}\n")
        if repl.done:
            return {"answer": repl.final, "steps": step + 1, "usage": total}
        history += [
            {"role": "assistant", "content": content},
            {"role": "user", "content": f"REPL output:\n{stdout}"},
        ]
    return {"answer": None, "steps": max_steps, "usage": total, "error": "max_steps exceeded"}


def _make_probe(context: Any) -> str:
    """生成 step 0 喂给模型的 context 形态描述（一并 print 到 REPL 里更好，但教学版简化）。"""
    if isinstance(context, dict):
        sizes = {k: (len(v) if hasattr(v, "__len__") else "?") for k, v in context.items()}
        return f"context type: dict\nkeys ({len(context)}): {list(context.keys())}\nvalue sizes: {sizes}"
    if isinstance(context, str):
        return f"context type: str, length={len(context)}\nfirst 300 chars: {context[:300]!r}"
    return f"context type: {type(context).__name__}"


### 4.4 v1 Demo：列出所有 paper_id

最 trivial 的测试——让模型证明它能写 `print(sorted(context.keys()))` 然后 `FINAL(...)`。


In [11]:
# 注意 question 怎么写：明确告诉模型"context 的 keys 就是 paper_id"。
# 写得含糊（"列出所有 paper_id"）模型会去 documents 里找 "paper_id" 字段，
# 这是 RLM 调试最常见的坑——任务描述要把数据形状讲清。
q1 = "把 context 的所有 keys 排序后返回（每个 key 就是一篇论文的 paper_id）。FINAL 一个 list[str]。"
result = await run_rlm_v1(q1, papers, max_steps=6, verbose=True)
print("\n=========== FINAL ===========")
print(result["answer"])
print(f"\nsteps={result['steps']}  usage={result['usage']}")



=== v1 step 0 ===
```repl
keys = sorted(context.keys())
FINAL(keys)
```

--- REPL ---
[EMPTY OUTPUT — remember to print() what you want to see]


=========== FINAL ===========
['Arbitrary_Lagrangian-Eulerian_and_Fluid-Structure Interaction Numerical Simulation (Benson)', 'PhD 基于通用程序的水下爆炸及其对结构作用的数值模拟研究', 'oezarmut_thyssenkrupp_Fluid-Composite Structure-Interaction in Underwater Shock Simulations', '不同加筋结构在水中接触爆炸下的破损规律', '基于ANSYS_LS-DYNA的钢板混凝土墙冲击实验的有限元分析', '基于LS-DYNA的液电效应冲击波数值模拟', '基于LS-DYNA的高速破片水中运动特性流固耦合数值模拟', '水下爆炸冲击荷载作用下混凝土重力坝的破坏模式']

steps=1  usage={'prompt_tokens': 823, 'completion_tokens': 16}


## §5 RLM v2：加 `await llm_query(...)` —— 异步递归子代理

v1 让模型自己 grep 太慢；它需要一个**副手**能吃掉一大块文本、自己再开 REPL 思考、
最后把答案交回。这正是 RLM 的"R"。

我们加一个 REPL 函数：

```python
async def llm_query(context, ...) -> Any:
    # 派生一个深度+1的子 RLM，给它新的 REPL，等它跑完
    # 子代理调 FINAL(value) 时，这个函数的返回值就是那个 value
```

设计要点：
- **`llm_query` 即递归**：它派生一个**完整的子 RLM**（包括它自己的 REPL 与历史），
  到达 `max_depth` 才退化为单次 LLM 调用。
- **子代理看不到父 REPL**：父代理必须把要给子代理看的内容**直接拼进 prompt**。
- 返回的是子代理传给 `FINAL(...)` 的真实 Python 对象，不是字符串化的形式。

这就是 fast-rlm 的方式（[reference prompt](https://github.com/avbiswas/fast-rlm/blob/main/src/prompt.ts)）。


In [13]:
MAX_DEPTH = 2  # depth=0 是根；到 MAX_DEPTH 时，llm_query 退化为直接的 LLM 调用

SYSTEM_V2 = """\
You are tasked with answering a query with associated context. You can access, transform, and analyze this context interactively in a REPL environment that can recursively query sub-LLMs, which you are strongly encouraged to use as much as possible. You will be queried iteratively until you provide a final answer.

You will be provided with information about your context by the user.
This metadata will include the context type, total characters, etc.

The REPL environment is initialized with:

1. A `context` variable that contains extremely important information about your query. You should check the content of the `context` variable to understand what you are working with.

   `context` may be a Python string OR a Python dict. When `context` is a dict, prefer indexing directly (e.g. `context["episodes"]`) instead of stringifying it.

2. A `llm_query` function that allows you to query a sub-LLM (with its own REPL!) inside your REPL environment. This function is asynchronous, so you must use `await llm_query(...)`. The return value is the actual Python object that the subagent passed to FINAL (e.g. a list, dict, string, etc.).

   Do NOT wrap the result in eval() or json.loads(); use it directly. Use python to minimize how much the LLM has to read.

3. A global function FINAL to return your answer as any native Python value.

** Understanding the level of detail user is asking for **
If the user wants exact details, be thorough. If the user wants a quick answer, prioritize speed. When you recursively spawn subagents, pass along the original intent if relevant.

You interact with the REPL by writing Python code wrapped in triple backticks with the `repl` language identifier.

Rules:
- print() statements show output (truncated to last 2000 chars).
- Just typing a variable name does NOT print it; use print().
- The REPL is persistent; past variables remain.
- Output exactly ONE ```repl``` block per turn.
- Do NOT use FINAL("variable_name") — pass the variable directly: FINAL(answer).

** How to control subagent behavior **
- If you want the subagent to return verbatim slices, instruct it to "slice the relevant sections and pass them to FINAL verbatim".
- If you want a summary, instruct it to summarize.
- Tell the subagent what shape you want back (list? dict? bullets? verbatim?).

** Iterative is OK **
This is multi-turn. You do not have to call FINAL on the first attempt. Print things first to inspect.
When a subagent returns, it is wise to print() the result and inspect before calling FINAL.

** Parallelism **
The biggest reason RLM is slow is running subagents one-after-the-other.
If you have multiple independent subtasks, kick them off in parallel with asyncio.gather:

```repl
import asyncio
tasks = [llm_query(f"...task A...") , llm_query(f"...task B...")]
a, b = await asyncio.gather(*tasks)
print(a, b)
```

** Example **

```repl
chunk = context[\"some_key\"][:30000]
answer = await llm_query(f\"Task: extract the authors' three main claims.\\n\\nPaper text:\\n{chunk}\")
print(answer)
```

When done, call FINAL(your_answer) in a repl block.

* WHAT IS BAD *
Reading the whole context piece by piece into your own history and then re-emitting it is a sign of low intelligence. Use python + llm_query.

* KNOWING WHEN TO QUIT *
Time is ticking. If you cannot finish, return what you know honestly.

Your response format:
```repl
your python code
FINAL(answer)
```
"""

SYSTEM_LEAF = """\
You are at the maximum recursion depth — you can no longer call llm_query, you must solve this on your own using the REPL.

The REPL has:
- `context` variable
- `FINAL(value)` to submit your answer
- `print()` to view output (truncated to last 2000 chars)

Write Python in a ```repl ... ``` block. Inspect the context, extract what you need, call FINAL.
"""


下面是支持 `llm_query` 的主循环。注意它和 v1 的差别：
1. 主循环改名 `run_rlm`（不带版本后缀，因为 v3 会扩展同一函数）
2. REPL 的 namespace 里注入 `llm_query`，闭包里捕获 `depth` 与 `max_depth`
3. 子代理把 `papers` 作为它自己的 `context` 吗？— **不**。父代理决定要把什么传下去。
   `llm_query(prompt)` 的 prompt 可以是字符串、dict、list；它会成为子代理 REPL 的 `context`。


In [14]:
async def run_rlm(question: str, context: Any, depth: int = 0,
                  max_steps: int = 10, max_depth: int = MAX_DEPTH, verbose: bool = True,
                  extra_tools: dict[str, Any] | None = None) -> dict:
    """通用 RLM 主循环；v2 = 没有 extra_tools；v3 会传 search_kb 进 extra_tools。"""
    is_leaf = depth >= max_depth
    repl = REPL(ns={"context": context})

    async def llm_query(sub_context: Any) -> Any:
        """在 REPL 里被调用：派生一个子 RLM 处理 sub_context（必须自带任务说明）。"""
        if is_leaf:
            # 到达最大深度——退化为单次 LLM 调用，不再有子 REPL
            prompt = sub_context if isinstance(sub_context, str) else json.dumps(sub_context, ensure_ascii=False)
            content, _ = await achat([{"role": "user", "content": prompt}])
            return content
        sub = await run_rlm(
            question="见 context 内的 task 描述",
            context=sub_context,
            depth=depth + 1,
            max_steps=max_steps,
            max_depth=max_depth,
            verbose=False,
            extra_tools=extra_tools,
        )
        return sub.get("answer")

    repl.ns["llm_query"] = llm_query
    if extra_tools:
        repl.ns.update(extra_tools)

    sys_prompt = SYSTEM_LEAF if is_leaf else SYSTEM_V2
    probe = _make_probe(context)
    if extra_tools and not is_leaf:
        tool_lines = "\n".join(f"  - {name}: {getattr(fn, '__doc__', None) or 'no docstring'}" for name, fn in extra_tools.items())
        probe += f"\n\nExtra tools in your REPL namespace:\n{tool_lines}"

    history = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"任务：{question}\n\nREPL 初始 probe:\n{probe}"},
    ]
    indent = "  " * depth
    total = {"prompt_tokens": 0, "completion_tokens": 0}
    for step in range(max_steps):
        content, usage = await achat(history)
        for k in total: total[k] += usage[k]
        if verbose:
            print(f"\n{indent}=== d={depth} s={step} ===\n{textwrap.indent(content, indent)}\n")
        code_str = extract_repl(content)
        if code_str is None:
            history += [
                {"role": "assistant", "content": content},
                {"role": "user", "content": "你的回复里没有 ```repl``` 代码块。请重写一个。"},
            ]
            continue
        stdout = await repl.execute(code_str)
        if verbose:
            print(f"{indent}--- REPL output ---\n{textwrap.indent(stdout, indent)}\n")
        if repl.done:
            return {"answer": repl.final, "steps": step + 1, "usage": total, "depth": depth}
        history += [
            {"role": "assistant", "content": content},
            {"role": "user", "content": f"REPL output:\n{stdout}"},
        ]
    return {"answer": None, "steps": max_steps, "usage": total, "depth": depth, "error": "max_steps"}


### 5.1 v2 Demo：用 `llm_query` 抽取单篇论文的关键字段

挑 oezarmut 那篇短英文论文（~30k 字符，单次 LLM 调用能吃下），让根代理 `llm_query`
一次拿到结构化字段。


In [15]:
q2 = """从 oezarmut 那篇关于 FRP 水下冲击的会议论文里抽取以下信息，返回一个 dict：
- title: 论文标题
- authors: 作者列表（list[str]）
- software: 用到的仿真软件
- key_finding: 一句话主要结论（中文）
最终 FINAL 一个 Python dict。"""

result = await run_rlm(q2, papers, max_steps=8, max_depth=2, verbose=True)
print("\n=========== FINAL ===========")
print(json.dumps(result["answer"], ensure_ascii=False, indent=2) if isinstance(result["answer"], (dict, list)) else result["answer"])
print(f"\nsteps={result['steps']}  usage={result['usage']}")



=== d=0 s=0 ===
I'll look at the paper by oezarmut about FRP underwater shock. The key paper seems to be `'oezarmut_thyssenkrupp_Fluid-Composite Structure-Interaction in Underwater Shock Simulations'`.

```repl
paper = context['oezarmut_thyssenkrupp_Fluid-Composite Structure-Interaction in Underwater Shock Simulations']
print(paper[:5000])
```

--- REPL output ---
[TRUNCATED: last 2000 chars]
...in the water is then transmitted radially outward as a compressive shock wave propagating with a velocity that is several times the speed of sound in uncompressed water. The velocity of the shock wave falls down to the acoustic level after reaching a distance of about 2-3 times the radius of the charge [2]. Fig. 2 illustrates an underwater explosion and the formation of the resulting steep shock front. As mentioned in [3], the pressure-density relationship becomes nonlinear for very high-pressure amplitudes leading to higher gradients of the pressure-density curve. Due to these higher gradient

### 5.2 信息瓶颈是 RLM 的核心 insight

注意 `llm_query` 的返回值是**子代理传给 FINAL 的那个 Python 对象**——而不是子代理
完整的对话历史，也不是子代理读过的全部 context。

如果子代理读了 30k 字符才得出一个 dict，父代理只看到那个 dict（也许 200 字符）。
**用变量隔离上下文**是 RLM 能处理超大语料的根本原因。

如果你想让父代理看到子代理的中间状态，那已经不是 RLM 了——你在写一个 agent framework。


## §6 RLM v3：加 `search_kb` 关键词工具

模型当前只能 `context["..."][:30000]` 硬切。对 8 篇论文的中英混合知识库，
更聪明的是先做关键词检索锁定相关论文/段落，再去精读。

我们暴露 `search_kb(keywords, top_k=5) -> list[dict]`：
对所有论文做不区分大小写的子串匹配，按命中数排序，返回片段。


In [16]:
def build_search_kb(papers: dict[str, str]):
    lowered = {pid: text.lower() for pid, text in papers.items()}

    def search_kb(keywords: list[str], top_k: int = 5, snippet_chars: int = 400) -> list[dict]:
        """Case-insensitive substring search across all loaded papers.

        Returns list of dicts sorted by total hit count:
            {"paper_id": str, "score": int, "hits": dict[str, int], "snippet": str}
        Use this BEFORE running expensive llm_query calls — narrows the search space.
        Pass multiple synonyms / acronyms (the corpus is bilingual).
        """
        kws = [k.lower() for k in keywords if k]
        if not kws: return []
        scored = []
        for pid, low in lowered.items():
            hits, first = {}, None
            for kw in kws:
                c = low.count(kw)
                if c > 0:
                    hits[kw] = c
                    pos = low.find(kw)
                    if first is None or pos < first: first = pos
            if hits:
                start = max(0, (first or 0) - snippet_chars // 2)
                end = min(len(papers[pid]), (first or 0) + snippet_chars // 2)
                snippet = papers[pid][start:end].replace("\n", " ")
                scored.append({"paper_id": pid, "score": sum(hits.values()), "hits": hits, "snippet": snippet})
        scored.sort(key=lambda r: -r["score"])
        return scored[:top_k]

    return search_kb

search_kb = build_search_kb(papers)
# 自测
print(json.dumps(search_kb(["LS-DYNA", "ALE", "流固耦合"], top_k=3), ensure_ascii=False, indent=2)[:800])


[
  {
    "paper_id": "Arbitrary_Lagrangian-Eulerian_and_Fluid-Structure Interaction Numerical Simulation (Benson)",
    "score": 244,
    "hits": {
      "ls-dyna": 3,
      "ale": 241
    },
    "snippet": ". . . . . . . . . . . . . . . . . . . . 45 Application to Dynamic Problems . . . . . . . . . . . 51 Mhamed SO U L I 2.1. Introduction . . . . . . . . . . . . . . . . . . . . . . . . . 51 2.2. General ALE description of Navier–Stokes equations . . . . . . . . . . . . . . . . . . . . . . . . . . . 54 2.3. Fluid–structure interaction . . . . . . . . . . . . . . 57 2.3.1. Contact algorithms for ﬂuid–s"
  },
  {
    "paper_id": "基于LS-DYNA的液电效应冲击波数值模拟",
    "score": 50,
    "hits": {
      "ls-dyna": 22,
      "ale": 28
    },
    "snippet": "DOI：10.11883/bzycj-2021-0214  # 基于 LS-DYNA 的液电效应


### 6.1 v3 Demo：用 search_kb 缩小范围再精读


In [17]:
q3 = "知识库里有几篇论文涉及 LS-DYNA？分别在做什么类型的仿真？用中文给出 paper_id 与一句话简介，最终 FINAL 一个 dict[paper_id, str]。"

EXTRA_TOOLS = {"search_kb": search_kb}
result = await run_rlm(q3, papers, max_steps=10, max_depth=2, verbose=True, extra_tools=EXTRA_TOOLS)
print("\n=========== FINAL ===========")
print(json.dumps(result["answer"], ensure_ascii=False, indent=2) if isinstance(result["answer"], (dict, list)) else result["answer"])
print(f"\nsteps={result['steps']}  usage={result['usage']}")



=== d=0 s=0 ===
```repl
# First, let's use search_kb to find papers mentioning "LS-DYNA" to confirm coverage
results = search_kb("LS-DYNA")
print(f"Number of papers mentioning LS-DYNA: {len(results)}")
for r in results:
    print(f"  {r['paper_id']}: score={r['score']}, snippet={r['snippet'][:100]}")
```

--- REPL output ---
Number of papers mentioning LS-DYNA: 5
  Arbitrary_Lagrangian-Eulerian_and_Fluid-Structure Interaction Numerical Simulation (Benson): score=116563, snippet=–Structure Interaction Arbitrary Lagrangian- Eulerian and Fluid  # Arbitrary Lagrangian Eulerian and
  oezarmut_thyssenkrupp_Fluid-Composite Structure-Interaction in Underwater Shock Simulations: score=7890, snippet=# Fluid-Composite Structure-Interaction in Underwater Shock Simulations  B. Özarmut1, A. Rühl1, B. H
  PhD 基于通用程序的水下爆炸及其对结构作用的数值模拟研究: score=7143, snippet=而只能获得少部分的物理 图景 。  过程参量数据 的丰富性 以获得大量 的过程量 的数据 以及 。通过数值模拟 可 变化 曲线以更加深刻地反映物理过程。但是对于水下爆炸实验来说往往只能获得少量数据
  基于LS-DYNA的液电效应冲击波数值模拟: score=3741, snippet=

## §7 端到端实战 Q&A

下面跑两个"非平凡"的问题，观察 RLM 是怎么组合 `search_kb + llm_query` 出答案的。

⚠️ 实战问题每个可能跑几十秒到几分钟（取决于递归深度和子代理是否并行）。

### 7.1 比较两篇文献的方法论


In [18]:
real_q1 = """比较 Benson 的 ALE/FSI 综述 (paper_id 含 'Benson') 与 PhD 学位论文 '基于通用程序的水下爆炸及其对结构作用的数值模拟研究' (paper_id 含 'PhD') 两篇文献：
它们对"流固耦合"这一概念的处理思路有何异同？请用中文作答，并在答案末尾标出引用的 paper_id。
最终 FINAL 一个字符串（markdown 格式）。"""

t0 = time.time()
result = await run_rlm(real_q1, papers, max_steps=12, max_depth=2, verbose=True, extra_tools=EXTRA_TOOLS)
print("\n=========== FINAL ===========\n")
print(result["answer"])
print(f"\nsteps={result['steps']}  usage={result['usage']}  time={time.time()-t0:.1f}s")



=== d=0 s=0 ===
```repl
import asyncio

# First, search for "流固耦合" in both papers to get relevant snippets
benson_paper = "Arbitrary_Lagrangian-Eulerian_and_Fluid-Structure Interaction Numerical Simulation (Benson)"
phd_paper = "PhD 基于通用程序的水下爆炸及其对结构作用的数值模拟研究"

# Search for FSI-related terms in both papers
tasks = [
    search_kb("流固耦合", [benson_paper, phd_paper]),
    search_kb("FSI", [benson_paper, phd_paper]),
    search_kb("fluid-structure", [benson_paper, phd_paper]),
    search_kb("ALE", [benson_paper, phd_paper])
]
results = await asyncio.gather(*tasks)
for r in results:
    print(f"Search result: {r}")
```

--- REPL output ---

Traceback (most recent call last):
  File "/tmp/ipykernel_558383/1556096464.py", line 23, in execute
    await result
  File "<repl>", line 9, in <module>
  File "/tmp/ipykernel_558383/3690237824.py", line 29, in search_kb
    return scored[:top_k]
           ~~~~~~^^^^^^^^
TypeError: slice indices must be integers or None or have an __index__ method





### 7.2 抽取一组论文里的关键参数


In [19]:
real_q2 = """在知识库里关于"加筋结构 / 钢板 / 混凝土墙 在水中接触爆炸 或 冲击 实验"的研究中，
作者们主要关注哪几个仿真/实验参数？这些参数对破口面积/形状/破坏模式 的影响是什么？
请用中文回答，并标出每条结论引用的 paper_id。最终 FINAL 一个字符串。"""

t0 = time.time()
result = await run_rlm(real_q2, papers, max_steps=12, max_depth=2, verbose=True, extra_tools=EXTRA_TOOLS)
print("\n=========== FINAL ===========\n")
print(result["answer"])
print(f"\nsteps={result['steps']}  usage={result['usage']}  time={time.time()-t0:.1f}s")



=== d=0 s=0 ===
```repl
# First, let's search for relevant papers about 加筋结构/钢板/混凝土墙 在水中接触爆炸/冲击
import asyncio

# Search for key terms
terms = ["加筋", "钢板混凝土墙", "接触爆炸", "冲击实验", "破口", "破坏模式"]
all_results = {}
for term in terms:
    results = search_kb(term)
    all_results[term] = results
    print(f"--- {term} ---")
    for r in results:
        print(f"  paper_id: {r['paper_id']}, score: {r['score']}, snippet: {r['snippet'][:200]}")
    print()
```

--- REPL output ---
[TRUNCATED: last 2000 chars]
...3/1001-1455(2014)05-0534-05  # 基于 LS-DYNA 的高速破片水中运动特性 流固耦合数值模拟*  康 德,严 平 (海军工程大学兵器工程系,湖北 武汉 430033)  摘要:基于大型有限元分析软件 ANSYS/LS-DYNA,建立三 维 长 方 体 高 速 破 片 在 水 介 质
  paper_id: 水下爆炸冲击荷载作用下混凝土重力坝的破坏模式, score: 75, snippet: 文章编号:1001-1455(2012)05-0501-07  # 水下爆炸冲击荷载作用下 混凝土重力坝的破坏模式*  张社荣,王高辉,王 超,孙 博 (天津大学水利工程仿真与安全国家重点实验室,天津 300072)  摘要:考虑混凝土的高应变率 效 应,构 建 重 力 坝 水 下 爆 炸 全 耦 合 模 型,运 用 显 式 动 力 分 析 程 序 LS-DY- NA,对水下爆炸冲击荷载作用下大 
  paper_id: 基于ANSYS_LS-DYNA的钢板混凝土墙冲击实验的有限元分析, score: 22, snippet: 学海岸与近海国家重点实验

## §8 这个教学版 RLM 的局限 & 思考题

### 当前局限（按"如果想做生产"的优先级排序）

1. **同进程 `exec()`** — LLM 写的 Python 没有任何沙箱。生产里换 Pyodide (fast-rlm) / Docker / E2B。
2. **无金额/token 硬截断** — 只有 `max_steps`/`max_depth`。生产里要按 token / 美元封顶（参看 fast-rlm 的 `max_money_spent`/`max_completion_tokens`/`max_prompt_tokens`）。
3. **`llm_query` 串行** — 教学版的 demo 都是串行的。fast-rlm 的 prompt 中鼓励 `asyncio.gather(*tasks)` 并行多个子代理；这里 REPL 已经支持 `await`，所以 LLM **可以**自己写 gather，但本 notebook 没强制示范。
4. **无 schema 校验** — `FINAL("不符合要求的英文")` 会通过。生产里加 Pydantic / JSON Schema（fast-rlm 已经做到了）。
5. **无历史压缩** — 长任务会越跑 history 越长。生产里加 compaction（参 rlms.RLM 的 `compaction=True`）。
6. **`search_kb` 太弱** — 只是子串。换成 BM25 / embedding 检索会更强。
7. **dubrify 后端 cost** — 我们打印 `usage = {prompt_tokens, completion_tokens}` 但不算钱。把 dubrify 的定价表抄下来按 token 估即可。

### 思考题

1. 在 §7.1 demo 里观察根代理是否真的用了 `llm_query`？如果它直接 `context["..."]` 切了就完成，说明给的问题"不够深"。设计一个一定需要递归的问题（比如"两两对比 8 篇论文的方法"）。
2. 把 `MAX_DEPTH=1` 改为 `MAX_DEPTH=3`，对同一个问题观察 token 用量和回答质量的变化。
3. 改写 `run_rlm` 使父代理在主循环里**并行**派发多个 `llm_query`（例如在 LLM 自己写出 `asyncio.gather` 时观察是否真的并行）。
4. 用 `text-embedding-3-small` 把 `search_kb` 改成 embedding 检索，对比关键词搜索的命中差异。
5. 给主循环加 `max_money_spent`：用 dubrify 的定价表，每次 chat 估一个 cost，超就抛错。

至此，你应当：
- 能解释 RLM = LLM + REPL + 递归子代理 的具体含义
- 能看着 fast-rlm 或 rlms 的源码找到本 notebook 的每个对应位置
- 知道在自己的问题上需要往哪些方向扩展
